# Live MLOps Demo: MLflow + Streamlit on Google Colab

**What students will build in this session**

| Stage | What happens | Tool |
|---|---|---|
| 1. Data | Load and inspect the dataset | scikit-learn, pandas |
| 2. Experiment tracking | Train several models, log params, metrics, plots | MLflow |
| 3. Model selection | Compare runs, choose the best one | MLflow |
| 4. Packaging | Save the model + metadata + training statistics | joblib, JSON |
| 5. Serving | Interactive web app | Streamlit |
| 6. Deployment | Public URL from Colab | Cloudflare tunnel |
| 7. Hot-swap | Deploy a new model version without restarting the app | |
| 8. Monitoring | Log predictions, detect data drift | Streamlit + pandas |

**How to run:** Runtime -> Run all, or go cell by cell during the lesson. Cells are re-runnable.

## 0. Setup

In [ ]:
%%capture
!pip install -q streamlit mlflow scikit-learn joblib pandas matplotlib requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import os, json, time, re, subprocess, warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

# ---- Project configuration (single place to change things) ----
EXPERIMENT_NAME = "iris-demo"
MODEL_DIR       = "models"
MODEL_PATH      = f"{MODEL_DIR}/model.joblib"
META_PATH       = f"{MODEL_DIR}/metadata.json"
STATS_PATH      = f"{MODEL_DIR}/train_stats.json"
LOG_DIR         = "logs"
PRED_LOG_PATH   = f"{LOG_DIR}/predictions.csv"
RANDOM_STATE    = 42

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Local SQLite backend: no server needed, works great in Colab
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow version :", mlflow.__version__)
print("Tracking URI   :", mlflow.get_tracking_uri())

## 1. Data: load and explore

We use the classic Iris dataset (150 flowers, 4 measurements, 3 species).
In real MLOps, this step would pull from a data warehouse or feature store.

In [ ]:
iris = load_iris(as_frame=True)
FEATURES = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
CLASSES  = list(iris.target_names)          # ['setosa', 'versicolor', 'virginica']

df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(CLASSES)))

print("Shape:", df.shape)
display(df.head())
display(df.describe().round(2))
print("\nClass balance:")
print(df["species"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for sp, g in df.groupby("species"):
    axes[0].scatter(g[FEATURES[0]], g[FEATURES[1]], label=sp, alpha=.7)
    axes[1].scatter(g[FEATURES[2]], g[FEATURES[3]], label=sp, alpha=.7)
axes[0].set(xlabel="sepal length", ylabel="sepal width", title="Sepal")
axes[1].set(xlabel="petal length", ylabel="petal width", title="Petal")
axes[0].legend()
plt.tight_layout(); plt.show()

## 2. Split the data

We **stratify** so each split has the same class balance. A fixed `random_state` makes runs reproducible,
which is essential for comparing experiments fairly.

In [ ]:
X = df[FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 3. Experiment tracking with MLflow

Each **run** records:
- **Parameters**: the settings we chose
- **Metrics**: how well the model did
- **Artifacts**: files such as plots and the model itself
- **Tags**: free-form labels

Wrapping training in a function lets us run many experiments with one line each.

In [ ]:
def train_and_log(n_estimators=100, max_depth=None, min_samples_split=2, run_name=None):
    """Train a RandomForest, log everything to MLflow, return (run_id, metrics)."""
    with mlflow.start_run(run_name=run_name) as run:
        params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth if max_depth is not None else -1,   # -1 means unlimited
            "min_samples_split": min_samples_split,
            "random_state": RANDOM_STATE,
            "test_size": 0.2,
        }
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=RANDOM_STATE,
        ).fit(X_train, y_train)

        y_pred = model.predict(X_test)
        metrics = {
            "accuracy":  accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="weighted"),
            "recall":    recall_score(y_test, y_pred, average="weighted"),
            "f1":        f1_score(y_test, y_pred, average="weighted"),
            "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.set_tags({"team": "mlops-class", "dataset": "iris", "model_family": "random_forest"})

        # Artifact 1: confusion matrix image
        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASSES, ax=ax, colorbar=False)
        ax.set_title(f"Confusion matrix ({run_name or run.info.run_id[:6]})")
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        # Artifact 2: the model, with a signature (expected input/output schema)
        # cloudpickle format avoids the skops "untrusted type" error
        signature = infer_signature(X_train, model.predict(X_train))
        mlflow.sklearn.log_model(
            model, name="model", signature=signature,
            input_example=X_train.head(3), serialization_format="cloudpickle",
        )
        return run.info.run_id, metrics

# First single run
run_id, m = train_and_log(n_estimators=100, run_name="baseline")
print("Run ID:", run_id)
print({k: round(v, 4) for k, v in m.items()})

### Hyperparameter sweep

Now we run a small grid. Each combination becomes its own tracked run, so nothing is lost
and every result is comparable.

In [ ]:
grid = [
    {"n_estimators": 5,   "max_depth": 2},
    {"n_estimators": 10,  "max_depth": 3},
    {"n_estimators": 50,  "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": None},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 4},
]

for cfg in grid:
    name = f"rf_n{cfg['n_estimators']}_d{cfg.get('max_depth')}"
    rid, m = train_and_log(run_name=name, **cfg)
    print(f"{name:<22} accuracy={m['accuracy']:.3f}  f1={m['f1']:.3f}")

## 4. Compare runs

In the MLflow UI you would click through runs. Here we do the same programmatically
with `mlflow.search_runs`, which is how automated pipelines pick a winner.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.accuracy DESC", "metrics.f1 DESC", "start_time DESC"],
)
cols = ["tags.mlflow.runName", "params.n_estimators", "params.max_depth",
        "metrics.accuracy", "metrics.f1", "metrics.train_accuracy", "run_id"]
display(runs[cols].round(4))

In [ ]:
plot_df = runs.sort_values("start_time")
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(plot_df["tags.mlflow.runName"], plot_df["metrics.accuracy"], color="#4C78A8")
ax.set_ylim(0.6, 1.02)
ax.set_ylabel("Test accuracy")
ax.set_title("Accuracy by run")
plt.xticks(rotation=40, ha="right")
plt.tight_layout(); plt.show()

**Discussion question for students:** several runs may tie on accuracy because the test set has only 30 rows.
How would you break the tie? (Hint: simpler model? faster inference? cross-validation?)

## 5. Package and "deploy" the best model

Deployment here means writing the model to a known location that the serving app reads.
Alongside the model we save:

- **metadata.json**: which run produced it, its metrics, its version number
- **train_stats.json**: mean and standard deviation of every feature at training time, used later for **drift detection**

In [ ]:
def deploy_run(run_id, note=""):
    """Load a model from MLflow by run_id and publish it to the serving folder."""
    model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
    run = mlflow.get_run(run_id)

    # Version number = previous version + 1
    version = 1
    if os.path.exists(META_PATH):
        version = json.load(open(META_PATH)).get("version", 0) + 1

    joblib.dump(model, MODEL_PATH)

    meta = {
        "version": version,
        "run_id": run_id,
        "run_name": run.data.tags.get("mlflow.runName", ""),
        "metrics": {k: round(v, 4) for k, v in run.data.metrics.items()},
        "params": run.data.params,
        "deployed_at": datetime.now().isoformat(timespec="seconds"),
        "note": note,
    }
    json.dump(meta, open(META_PATH, "w"), indent=2)

    stats = {f: {"mean": float(X_train[f].mean()), "std": float(X_train[f].std())} for f in FEATURES}
    json.dump(stats, open(STATS_PATH, "w"), indent=2)

    print(f"Deployed version {version} from run '{meta['run_name']}'  (accuracy={meta['metrics']['accuracy']})")
    return meta

best_run_id = runs.iloc[0]["run_id"]
meta = deploy_run(best_run_id, note="initial deployment")
print(json.dumps(meta, indent=2))

In [ ]:
# Sanity check: load the deployed artifact exactly as the app will
served = joblib.load(MODEL_PATH)
sample = X_test.head(5)
print("Predictions:", served.predict(sample))
print("Actual     :", y_test.head(5).values)

## 6. The Streamlit app

The app has three tabs:

1. **Predict**: sliders, prediction, confidence, every prediction is logged to a CSV
2. **Monitoring**: prediction counts, confidence trend, feature drift vs. training data
3. **Model info**: version, metrics, parameters

The app checks the model file's modification time, so when we redeploy a new version
it is picked up automatically (no restart).

In [ ]:
%%writefile app.py
import os, json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import streamlit as st

MODEL_PATH    = "models/model.joblib"
META_PATH     = "models/metadata.json"
STATS_PATH    = "models/train_stats.json"
PRED_LOG_PATH = "logs/predictions.csv"
FEATURES = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
CLASSES  = ["setosa", "versicolor", "virginica"]
LOG_COLS = ["timestamp"] + FEATURES + ["prediction", "confidence", "model_version"]

st.set_page_config(page_title="Iris MLOps Demo", page_icon="🌸", layout="wide")


@st.cache_resource
def load_model(mtime):
    """mtime is part of the cache key, so a redeployed file reloads automatically."""
    return joblib.load(MODEL_PATH)


def load_json(path):
    return json.load(open(path)) if os.path.exists(path) else {}


def log_prediction(row):
    os.makedirs("logs", exist_ok=True)
    pd.DataFrame([row], columns=LOG_COLS).to_csv(
        PRED_LOG_PATH, mode="a", header=not os.path.exists(PRED_LOG_PATH), index=False
    )


if not os.path.exists(MODEL_PATH):
    st.error("No model deployed yet. Run the deployment cell in the notebook.")
    st.stop()

model = load_model(os.path.getmtime(MODEL_PATH))
meta  = load_json(META_PATH)
stats = load_json(STATS_PATH)

st.title("🌸 Iris Classifier")
st.caption(f"Serving model **v{meta.get('version', '?')}** ({meta.get('run_name', '')}) "
           f"| deployed {meta.get('deployed_at', '?')}")

tab_predict, tab_monitor, tab_info = st.tabs(["🔮 Predict", "📈 Monitoring", "ℹ️ Model info"])

# ------------------------------------------------------------------ Predict
with tab_predict:
    left, right = st.columns(2)
    with left:
        st.subheader("Flower measurements")
        sl = st.slider("Sepal length (cm)", 4.0, 8.0, 5.8, 0.1)
        sw = st.slider("Sepal width (cm)",  2.0, 4.5, 3.0, 0.1)
        pl = st.slider("Petal length (cm)", 1.0, 7.0, 4.3, 0.1)
        pw = st.slider("Petal width (cm)",  0.1, 2.5, 1.3, 0.1)

    x = pd.DataFrame([[sl, sw, pl, pw]], columns=FEATURES)
    proba = model.predict_proba(x)[0]
    pred = int(np.argmax(proba))

    with right:
        st.subheader("Prediction")
        st.success(f"**{CLASSES[pred]}**  ({proba[pred]:.1%} confident)")
        st.bar_chart(pd.DataFrame({"probability": proba}, index=CLASSES))
        if st.button("📝 Log this prediction"):
            log_prediction([datetime.now().isoformat(timespec="seconds"), sl, sw, pl, pw,
                            CLASSES[pred], round(float(proba[pred]), 4), meta.get("version", 0)])
            st.toast("Logged!")

# --------------------------------------------------------------- Monitoring
with tab_monitor:
    st.subheader("Production monitoring")
    if not os.path.exists(PRED_LOG_PATH):
        st.info("No predictions logged yet. Use the Predict tab and click 'Log this prediction'.")
    else:
        logs = pd.read_csv(PRED_LOG_PATH)
        c1, c2, c3 = st.columns(3)
        c1.metric("Total predictions", len(logs))
        c2.metric("Avg confidence", f"{logs['confidence'].mean():.1%}")
        c3.metric("Latest model version", int(logs["model_version"].iloc[-1]))

        st.markdown("**Predictions by class**")
        st.bar_chart(logs["prediction"].value_counts())

        st.markdown("**Confidence over time** (rolling mean of 10)")
        st.line_chart(logs["confidence"].rolling(10, min_periods=1).mean())

        st.markdown("**Feature drift** (z-score of recent mean vs. training mean)")
        window = st.slider("Use last N predictions", 10, max(len(logs), 11), min(len(logs), 50))
        recent = logs.tail(window)
        z = pd.Series({f: (recent[f].mean() - stats[f]["mean"]) / stats[f]["std"] for f in FEATURES})
        st.bar_chart(z.rename("z-score"))
        drifted = z[z.abs() > 1.0]
        if len(drifted):
            st.warning("Possible data drift in: " + ", ".join(drifted.index) +
                       ". Consider investigating and retraining.")
        else:
            st.success("No significant drift detected.")

        with st.expander("Raw log"):
            st.dataframe(logs.tail(50), use_container_width=True)

# --------------------------------------------------------------- Model info
with tab_info:
    st.subheader("Deployed model")
    st.json(meta)

## 7. Launch the app and get a public URL

Colab does not expose ports directly, so we use a **Cloudflare quick tunnel**.
The helper below waits until the tunnel is actually registered before printing the URL
(printing too early is the usual reason a link "doesn't work").

In [ ]:
def stop_all():
    subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
    time.sleep(2)

def wait_for_port(port, timeout=40):
    for _ in range(timeout):
        try:
            if requests.get(f"http://localhost:{port}", timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(1)
    return False

def start_tunnel(port, logfile, timeout=90, rewrite_host=False):
    """Start a cloudflared quick tunnel and return its public URL once registered.
    rewrite_host=True makes cloudflared send Host: localhost:<port> to the local server
    (needed for MLflow, which rejects unknown hosts as a DNS-rebinding protection)."""
    host_flag = f"--http-host-header localhost:{port}" if rewrite_host else ""
    subprocess.Popen(f"./cloudflared tunnel --url http://localhost:{port} {host_flag} > {logfile} 2>&1", shell=True)
    for _ in range(timeout):
        time.sleep(1)
        log = open(logfile).read() if os.path.exists(logfile) else ""
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if m and "Registered tunnel connection" in log:
            time.sleep(10)   # let DNS propagate
            return m.group(0)
    print("Tunnel did not register. Last log lines:\n", "\n".join(log.splitlines()[-8:]))
    return None

stop_all()
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1", shell=True)

if wait_for_port(8501):
    print("Streamlit is running locally.")
    APP_URL = start_tunnel(8501, "tunnel_app.log")
    print("\n>>> Share this link with students:", APP_URL)
else:
    print("Streamlit failed to start. Log:\n", open("streamlit.log").read()[-1500:])

**If the link does not open:** wait 30-60 seconds and refresh, try an incognito window or phone data,
or switch DNS to 1.1.1.1. Fallback for yourself only:

```python
from google.colab import output
output.serve_kernel_port_as_window(8501)
```

## 8. Live demo: hot-swap a new model version

While the app is open on students' screens, we train a different model and deploy it.
After you click anything in the app (or refresh), the header shows the new version.

In [ ]:
# Train a deliberately different model (a tiny, weak one) so the change is obvious
weak_id, m = train_and_log(n_estimators=3, max_depth=1, run_name="rf_tiny_demo")
print("Weak model accuracy:", round(m["accuracy"], 3))

deploy_run(weak_id, note="demo: deploying a weaker model on purpose")

In [ ]:
# Rollback: redeploy the best run we found earlier
deploy_run(best_run_id, note="rollback to best model")

**Talking points:** versioned artifacts, rollback in one line, why you gate deployments on metrics
(a CI pipeline would refuse to deploy `rf_tiny_demo`).

### Automated deployment gate (mini CI/CD)

A real pipeline would only deploy a candidate that beats the current model.

In [ ]:
def deploy_if_better(candidate_run_id, min_gain=0.0):
    current = json.load(open(META_PATH))["metrics"]["accuracy"]
    cand = mlflow.get_run(candidate_run_id).data.metrics["accuracy"]
    print(f"current={current:.3f}  candidate={cand:.3f}")
    if cand > current + min_gain:
        return deploy_run(candidate_run_id, note="auto-deployed: beat current model")
    print("Rejected: candidate does not beat the current model.")

deploy_if_better(weak_id)      # should be rejected

## 9. Monitoring: simulate production traffic and drift

We append synthetic requests to the prediction log in two phases:

1. **Normal traffic** resembling the training data
2. **Drifted traffic** where petals are much larger (imagine a new sensor or a new species)

Then open the app's **Monitoring** tab and watch the drift chart.

In [ ]:
def simulate_traffic(n=60, shift=0.0, seed=0):
    """Sample rows from the real data, add a shift to the petal features, predict, and log."""
    rng = np.random.default_rng(seed)
    model = joblib.load(MODEL_PATH)
    version = json.load(open(META_PATH))["version"]

    rows = df.sample(n, replace=True, random_state=seed)[FEATURES].copy()
    rows[FEATURES[2]] += shift + rng.normal(0, 0.1, n)
    rows[FEATURES[3]] += shift / 3 + rng.normal(0, 0.05, n)

    proba = model.predict_proba(rows)
    out = rows.copy()
    out.insert(0, "timestamp", datetime.now().isoformat(timespec="seconds"))
    out["prediction"] = [CLASSES[i] for i in proba.argmax(1)]
    out["confidence"] = proba.max(1).round(4)
    out["model_version"] = version

    out.to_csv(PRED_LOG_PATH, mode="a", header=not os.path.exists(PRED_LOG_PATH), index=False)
    print(f"Logged {n} predictions (shift={shift})")

if os.path.exists(PRED_LOG_PATH):
    os.remove(PRED_LOG_PATH)            # start fresh

simulate_traffic(n=60, shift=0.0, seed=1)    # normal

In [ ]:
# Now inject drift, then refresh the Monitoring tab
simulate_traffic(n=60, shift=1.5, seed=2)

In [ ]:
# Same drift calculation in the notebook, so students see the logic behind the chart
logs = pd.read_csv(PRED_LOG_PATH)
stats = json.load(open(STATS_PATH))
recent = logs.tail(50)
z = pd.Series({f: (recent[f].mean() - stats[f]["mean"]) / stats[f]["std"] for f in FEATURES})
display(z.round(2).to_frame("z-score"))
print("Drifted features:", list(z[z.abs() > 1].index))

## 10. Optional: show the MLflow UI

Launches the MLflow server and tunnels it so students can browse experiments in the browser.

In [ ]:
subprocess.run("pkill -f 'mlflow ui'; pkill -f gunicorn", shell=True)   # clear any old MLflow server
time.sleep(2)
subprocess.Popen("mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000 --host 127.0.0.1 > mlflow_ui.log 2>&1", shell=True)
if wait_for_port(5000, timeout=60):
    # rewrite_host=True fixes: "Invalid Host header - possible DNS rebinding attack detected"
    MLFLOW_URL = start_tunnel(5000, "tunnel_mlflow.log", rewrite_host=True)
    print("MLflow UI:", MLFLOW_URL)
else:
    print(open("mlflow_ui.log").read()[-1500:])

## 11. Student exercises

1. Add a second model family (e.g. `LogisticRegression`) and compare it to the random forest in MLflow.
2. Use 5-fold cross-validation and log the mean and std of accuracy.
3. Add a "Download predictions CSV" button to the app.
4. Make `deploy_if_better` use F1 instead of accuracy.
5. Add a new monitoring metric: percentage of predictions with confidence below 60%.
6. Containerize the app with a `Dockerfile` (needs a local machine, not Colab).

## 12. Cleanup

In [ ]:
stop_all()
print("Stopped Streamlit and tunnels. Files remain in the Colab session until it is recycled.")
# To remove everything:  !rm -rf mlflow.db mlruns models logs artifacts app.py

## Recap: how this maps to production MLOps

| This notebook | In production |
|---|---|
| SQLite MLflow | MLflow server + Postgres + S3/GCS artifact store |
| `deploy_run()` copies a file | Model registry stages, container image, Kubernetes rollout |
| `deploy_if_better()` | CI/CD pipeline with tests and approval gates |
| Streamlit on a tunnel | FastAPI/Streamlit behind a load balancer |
| CSV prediction log | Logging pipeline + monitoring stack (Evidently, Grafana, etc.) |
| Manual drift z-score | Scheduled drift jobs and alerts that trigger retraining |